<a href="https://colab.research.google.com/github/hayazaqout/Gaza-Damage-Project/blob/main/Copy_of_Gaza_Damage_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ═══════════════════════════════════════════════
# CELL 0 — تثبيت المكتبات + تحميل الداتا
# ═══════════════════════════════════════════════

# تثبيت المكتبات
!pip install segmentation-models-pytorch albumentations kaggle osmnx rasterio geopandas -q

# تحميل الداتا من Kaggle
import os
from google.colab import files
# بنرفع ملف kaggle.json
uploaded = files.upload()
os.makedirs('/root/.config/kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.config/kaggle/kaggle.json')
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)

# حمّل الداتا
!kaggle datasets download -d abdoomoh/gaza-before-and-after-2 -p /content/data --unzip



In [ ]:
import pickle, os
SAVE_PATH = '/content/drive/MyDrive/Shati_Project'
if os.path.exists(f'{SAVE_PATH}/all_meta.pkl'):
    with open(f'{SAVE_PATH}/all_meta.pkl', 'rb') as f:
        all_meta = pickle.load(f)
    with open(f'{SAVE_PATH}/shati_pairs.pkl', 'rb') as f:
        pairs = pickle.load(f)
    print(f"محمّل من Drive — {len(pairs)} زوج جاهز!")
else:
    print("مش موجود ")

In [ ]:
# ═══════════════════════════════════════════════
# CELL 1 — استكشاف الداتا + تجهيز أزواج مخيم الشاطئ
# ═══════════════════════════════════════════════

import json
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict
import pickle
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = Path('/content/data/Gaza Before and After')
SAVE_PATH = '/content/drive/MyDrive/Shati_Project'
os.makedirs(SAVE_PATH, exist_ok=True)
CUTOFF = '2023-10-07'

# إحداثيات مخيم الشاطئ
SHATI = {'west': 34.425, 'south': 31.515, 'east': 34.465, 'north': 31.545}

# ─── دوال مساعدة ─────────────────────────────────
def is_valid(path):
    img = cv2.imread(str(path))
    if img is None: return False
    if img.mean() < 10: return False
    return True

def in_shati(bbox):
    try:
        return (bbox[0] >= SHATI['west']  - 0.05 and
                bbox[2] <= SHATI['east']  + 0.05 and
                bbox[1] >= SHATI['south'] - 0.05 and
                bbox[3] <= SHATI['north'] + 0.05)
    except:
        return False

# ─── 1. اجمع الـ metadata ────────────────────────
print("جاري قراءة الـ metadata...")
all_meta = []
for json_file in DATA_PATH.glob('**/*.json'):
    try:
        with open(json_file) as f:
            meta = json.load(f)
        section  = meta['section_id']
        date     = meta['timestamp']
        img_path = DATA_PATH / 'images' / section / f'{date}.png'
        meta['img_path']   = str(img_path)
        # بنضيف  bbox افتراضي لو ما موجود
        if 'bbox' not in meta:
            meta['bbox'] = [0, 0, 0, 0]
        meta['img_exists'] = img_path.exists() and is_valid(img_path)
        all_meta.append(meta)
    except:
        continue

all_meta.sort(key=lambda x: x['timestamp'])
total  = len(all_meta)
valid  = sum(1 for m in all_meta if m['img_exists'])
print(f"إجمالي: {total} | صالحة: {valid}")

# ─── 2. بنفلتر مخيم الشاطئ ────────────────────────
shati_meta = [
    m for m in all_meta
    if m['img_exists'] and m.get('bbox') and in_shati(m['bbox'])
]
print(f"صور مخيم الشاطئ: {len(shati_meta)}")

# لو ما في — بنوسع النطاق
if len(shati_meta) < 5:
    print("قليلة — بنوسّع النطاق...")
    shati_meta = [
        m for m in all_meta
        if m['img_exists']
        and m['bbox'] != [0,0,0,0]
        and abs((m['bbox'][0]+m['bbox'][2])/2 - 34.44) < 0.15
        and abs((m['bbox'][1]+m['bbox'][3])/2 - 31.53) < 0.12
    ]
    print(f" بعد التوسيع: {len(shati_meta)}")

# لو لسا ما في — بنوخد كل الداتا
if len(shati_meta) == 0:
    print("ما في صور مخيم الشاطئ — بنستخدم كل الداتا")
    shati_meta = [m for m in all_meta if m['img_exists']]

# ─── 3. قسّم pre / post ──────────────────────────
pre_meta  = [m for m in shati_meta if m['timestamp'] <  CUTOFF]
post_meta = [m for m in shati_meta if m['timestamp'] >= CUTOFF]

print(f"\npre  (قبل الحرب): {len(pre_meta)}")
print(f"post (بعد الحرب): {len(post_meta)}")

# ─── 4. بنجهز أفضل زوج لكل section ───────────────
pre_by_section  = defaultdict(list)
post_by_section = defaultdict(list)
for m in pre_meta:  pre_by_section[m['section_id']].append(m)
for m in post_meta: post_by_section[m['section_id']].append(m)

pairs = []
for section_id in pre_by_section:
    if section_id not in post_by_section:
        continue
    best_pre  = max(pre_by_section[section_id],  key=lambda x: x['timestamp'])
    best_post = min(post_by_section[section_id], key=lambda x: x['timestamp'])
    pairs.append({
        'section': section_id,
        'pre':     best_pre,
        'post':    best_post
    })

print(f"\nأزواج جاهزة: {len(pairs)}")

# ─── 5. بنعرض الأزواج ────────────────────────────
if pairs:
    show = min(len(pairs), 4)
    fig, axes = plt.subplots(show, 2, figsize=(10, 5*show))
    if show == 1: axes = [axes]
    for i in range(show):
        pair     = pairs[i]
        pre_img  = cv2.cvtColor(cv2.imread(pair['pre']['img_path']),  cv2.COLOR_BGR2RGB)
        post_img = cv2.cvtColor(cv2.imread(pair['post']['img_path']), cv2.COLOR_BGR2RGB)
        axes[i][0].imshow(pre_img)
        axes[i][0].set_title(f"Pre — {pair['pre']['timestamp']}\n{pair['section']}", fontsize=10)
        axes[i][0].axis('off')
        axes[i][1].imshow(post_img)
        axes[i][1].set_title(f"Post — {pair['post']['timestamp']}", fontsize=10)
        axes[i][1].axis('off')
    plt.suptitle("AL-shati camp— paires Pre/Post", fontsize=13)
    plt.tight_layout()
    plt.savefig("shati_pairs.png", dpi=150, bbox_inches='tight')
    plt.show()

# ─── 6. حفظ ────────────────────────────────────
with open(f'{SAVE_PATH}/shati_pairs.pkl', 'wb') as f:
    pickle.dump(pairs, f)

with open(f'{SAVE_PATH}/all_meta.pkl', 'wb') as f:
    pickle.dump(all_meta, f)

print(f"\n محفوظ: {len(pairs)} زوج على Drive!")
if pairs:
    print(f" أقدم pre:  {min(p['pre']['timestamp']  for p in pairs)}")
    print(f" أحدث post: {max(p['post']['timestamp'] for p in pairs)}")

In [ ]:
# ═══════════════════════════════════════════════
# CELL 2 — تجهيز بيانات تحديد المباني
# ═══════════════════════════════════════════════

import cv2
import numpy as np
import json
import os
import pickle
import osmnx as ox
import matplotlib.pyplot as plt
from pathlib import Path

SAVE_PATH = '/content/drive/MyDrive/Shati_Project'
DATA_PATH = Path('/content/data/Gaza Before and After')

# ─── بنحمل الأزواج ───────────────────────────────
with open(f'{SAVE_PATH}/shati_pairs.pkl', 'rb') as f:
    pairs = pickle.load(f)
print(f" {len(pairs)} زوج محمّل!")

# ─── 1. بنحمل مباني مخيم الشاطئ من OSM ──────────
print("تحميل المباني من OSM...")
buildings = ox.features_from_bbox(
    bbox=(34.415, 31.510, 34.475, 31.550),
    tags={'building': True}
)
buildings = buildings[
    buildings.geometry.geom_type.isin(['Polygon','MultiPolygon'])
].copy().to_crs('EPSG:4326').reset_index(drop=True)
print(f"{len(buildings)} مبنى من OSM!")

# ─── 2. دوال مساعدة ──────────────────────────────
def geo_to_pixel(lon, lat, bbox, w, h):
    x = int((lon - bbox[0]) / (bbox[2] - bbox[0]) * w)
    y = int((bbox[3] - lat) / (bbox[3] - bbox[1]) * h)
    return max(0,min(x,w-1)), max(0,min(y,h-1))

def make_building_mask(buildings, bbox, w, h):
    mask = np.zeros((h, w), dtype=np.uint8)
    for _, row in buildings.iterrows():
        try:
            geom  = row.geometry
            polys = list(geom.geoms) if geom.geom_type=='MultiPolygon' else [geom]
            for poly in polys:
                pts = np.array([
                    geo_to_pixel(lon, lat, bbox, w, h)
                    for lon, lat in poly.exterior.coords
                ], dtype=np.int32)
                cv2.fillPoly(mask, [pts], 1)
        except:
            continue
    return mask

# ─── 3. جهّزي الـ patches ────────────────────────
TARGET_SIZE = 256
STRIDE      = 64    # تداخل كبير عشان نحصل patches أكثر
MAX_PATCHES = 3000

all_images = []
all_masks  = []

# استخدمي كل صور pre لمخيم الشاطئ مش بس الزوج
from collections import defaultdict
with open(f'{SAVE_PATH}/all_meta.pkl', 'rb') as f:
    all_meta = pickle.load(f)

CUTOFF = '2023-10-07'
SHATI  = {'west':34.415, 'south':31.510, 'east':34.475, 'north':31.550}

def in_shati(bbox):
    try:
        cx = (bbox[0]+bbox[2])/2
        cy = (bbox[1]+bbox[3])/2
        return (SHATI['west']  <= cx <= SHATI['east'] and
                SHATI['south'] <= cy <= SHATI['north'])
    except:
        return False

# كل صور pre لمخيم الشاطئ
shati_pre = [
    m for m in all_meta
    if m['img_exists']
    and m['timestamp'] < CUTOFF
    and m.get('bbox') and m['bbox'] != [0,0,0,0]
    and in_shati(m['bbox'])
]
print(f"صور pre لمخيم الشاطئ: {len(shati_pre)}")

print(f"تجهيز الـ patches...")

for meta in shati_pre:
    if len(all_images) >= MAX_PATCHES:
        break

    img  = cv2.cvtColor(cv2.imread(meta['img_path']), cv2.COLOR_BGR2RGB)
    bbox = meta['bbox']
    h, w = img.shape[:2]

    # اعملنا  mask المباني
    bldg_mask = make_building_mask(buildings, bbox, w, h)

    # لو ما في مباني — تجاهل
    if bldg_mask.sum() < 500:
        continue

    # نقطع
    for y in range(0, h - TARGET_SIZE + 1, STRIDE):
        for x in range(0, w - TARGET_SIZE + 1, STRIDE):
            img_p  = img      [y:y+TARGET_SIZE, x:x+TARGET_SIZE]
            mask_p = bldg_mask[y:y+TARGET_SIZE, x:x+TARGET_SIZE]
            # بنوخد  patches فيها مباني كافية
            if mask_p.sum() > 200:
                all_images.append(img_p.astype(np.float32)/255.0)
                all_masks.append(mask_p.astype(np.float32))

    if len(all_images) % 300 == 0 and len(all_images) > 0:
        print(f"  patches: {len(all_images)}")

print(f"\n إجمالي الـ patches: {len(all_images)}")
print(f"   نسبة المباني:       {np.mean([m.mean() for m in all_masks])*100:.1f}%")

# ─── 4. هنا بنحفظ في الدرايف  ────────────────────────────────────
np.save(f'{SAVE_PATH}/building_images.npy', np.array(all_images, dtype=np.float32))
np.save(f'{SAVE_PATH}/building_masks.npy',  np.array(all_masks,  dtype=np.float32))


# ─── 5.  مثال ───────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for i in range(2):
    idx = i * 50 + 10
    overlay = all_images[idx].copy()
    mask_3ch = np.stack([all_masks[idx]]*3, axis=-1)
    yellow = np.array([1.0, 0.84, 0.0], dtype=np.float32)
    for c in range(3):
        overlay[:,:,c][all_masks[idx] > 0] = (
            overlay[:,:,c][all_masks[idx] > 0] * 0.4 + yellow[c] * 0.6
        )

    axes[i][0].imshow(all_images[idx])
    axes[i][0].set_title("The original image"); axes[i][0].axis('off')

    axes[i][1].imshow(all_masks[idx], cmap='gray')
    axes[i][1].set_title("Mask Buildings (OSM)"); axes[i][1].axis('off')

    axes[i][2].imshow(overlay)
    axes[i][2].set_title("The buildings are in yellow"); axes[i][2].axis('off')

plt.suptitle(f"Building Detection Data — {len(all_images)} patch", fontsize=13)
plt.tight_layout()
plt.savefig("building_data.png", dpi=150, bbox_inches='tight')
plt.show()